# Day 10: Beginner RNN and LSTM Lab with Movie Reviews

In this beginner lab, we will build simple neural networks for movie review sentiment classification.

Dataset: IMDB movie reviews from `tensorflow.keras.datasets.imdb`

Goal: predict whether a movie review is positive or negative.

What you will learn:
- how to load a text dataset from Keras,
- why text is converted into numbers,
- why padding is needed,
- how to build a basic RNN model,
- how to build a basic LSTM model,
- how to train and test both models.

## 1. Import Libraries

Run this cell first. If TensorFlow is not installed, install it with `pip install tensorflow` and restart Jupyter.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense

print("TensorFlow version:", tf.__version__)

## 2. Set Beginner-Friendly Values

These values keep the lab small and easier to run on a normal laptop.

In [ ]:
VOCAB_SIZE = 5000      # Use only the 5000 most common words
MAX_LENGTH = 100       # Every review will become 100 words/tokens long
TRAIN_SIZE = 8000      # Use a smaller training set for faster learning
TEST_SIZE = 2000       # Use a smaller test set for faster evaluation
BATCH_SIZE = 128
EPOCHS = 2             # Increase this later if you want better accuracy

## 3. Load the IMDB Movie Review Dataset

Keras gives this dataset already converted into numbers.

Each review is a list of word IDs.

Labels:
- `0` means negative review
- `1` means positive review

In [ ]:
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

# Keep smaller data for beginner practice
x_train = x_train[:TRAIN_SIZE]
y_train = y_train[:TRAIN_SIZE]
x_test = x_test[:TEST_SIZE]
y_test = y_test[:TEST_SIZE]

print("Training reviews:", len(x_train))
print("Testing reviews:", len(x_test))
print("First review as numbers:", x_train[0][:20])
print("First label:", y_train[0])

## 4. Understand Review Lengths

Reviews have different lengths. Neural networks need equal-length inputs, so we will fix this using padding.

In [ ]:
review_lengths = [len(review) for review in x_train]

print("Shortest review length:", min(review_lengths))
print("Average review length:", int(np.mean(review_lengths)))
print("Longest review length:", max(review_lengths))

plt.figure(figsize=(8, 4))
plt.hist(review_lengths, bins=40)
plt.title("Movie Review Lengths")
plt.xlabel("Number of words/tokens")
plt.ylabel("Number of reviews")
plt.show()

## 5. Pad the Reviews

Padding makes every review the same length.

Example:
- short reviews get zeros added,
- long reviews are cut down.

After this step, every review will have exactly `MAX_LENGTH` numbers.

In [ ]:
x_train_pad = pad_sequences(x_train, maxlen=MAX_LENGTH, padding="post", truncating="post")
x_test_pad = pad_sequences(x_test, maxlen=MAX_LENGTH, padding="post", truncating="post")

print("Training shape:", x_train_pad.shape)
print("Testing shape:", x_test_pad.shape)
print("First padded review:", x_train_pad[0][:30])

## 6. Decode One Review into Words

This step is only for understanding. The model will still use numbers, not words.

In [ ]:
word_index = imdb.get_word_index()
reverse_word_index = {value + 3: key for key, value in word_index.items()}
reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNKNOWN>"
reverse_word_index[3] = "<UNUSED>"


def decode_review(encoded_review):
    words = [reverse_word_index.get(number, "?") for number in encoded_review]
    return " ".join(words)

print("Decoded review sample:")
print(decode_review(x_train[0])[:600])
print()
print("Label:", "Positive" if y_train[0] == 1 else "Negative")

## 7. Build a Simple RNN Model

A Simple RNN reads the review step by step.

Model flow:
1. `Embedding`: turns word IDs into small vectors.
2. `SimpleRNN`: reads the sequence.
3. `Dense`: predicts positive or negative.

In [ ]:
rnn_model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=32, input_length=MAX_LENGTH),
    SimpleRNN(32),
    Dense(1, activation="sigmoid")
])

rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_model.summary()

## 8. Train the Simple RNN Model

During training, watch `accuracy` and `val_accuracy`.

Validation accuracy tells us how the model is doing on unseen validation data.

In [ ]:
rnn_history = rnn_model.fit(
    x_train_pad,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2
)

## 9. Test the Simple RNN Model

In [ ]:
rnn_loss, rnn_accuracy = rnn_model.evaluate(x_test_pad, y_test)
print("Simple RNN Test Accuracy:", round(rnn_accuracy, 4))

## 10. Build an LSTM Model

LSTM is an improved RNN. It is better at remembering useful information from earlier words in a sequence.

For beginners, remember this simple idea:
- RNN has short memory.
- LSTM has better memory.

In [ ]:
lstm_model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=32, input_length=MAX_LENGTH),
    LSTM(32),
    Dense(1, activation="sigmoid")
])

lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_model.summary()

## 11. Train the LSTM Model

In [ ]:
lstm_history = lstm_model.fit(
    x_train_pad,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2
)

## 12. Test the LSTM Model

In [ ]:
lstm_loss, lstm_accuracy = lstm_model.evaluate(x_test_pad, y_test)
print("LSTM Test Accuracy:", round(lstm_accuracy, 4))

## 13. Compare RNN and LSTM

In [ ]:
results = pd.DataFrame({
    "Model": ["Simple RNN", "LSTM"],
    "Test Accuracy": [round(rnn_accuracy, 4), round(lstm_accuracy, 4)]
})

results

## 14. Plot Accuracy

This graph helps us compare training and validation accuracy.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(rnn_history.history["accuracy"], label="RNN train")
plt.plot(rnn_history.history["val_accuracy"], label="RNN validation")
plt.plot(lstm_history.history["accuracy"], label="LSTM train")
plt.plot(lstm_history.history["val_accuracy"], label="LSTM validation")
plt.title("RNN vs LSTM Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## 15. Try a Simple Prediction

We will take one test review and ask the LSTM model to predict if it is positive or negative.

In [ ]:
sample_number = 0
sample_review = x_test_pad[sample_number:sample_number + 1]
actual_label = y_test[sample_number]

prediction = lstm_model.predict(sample_review)[0][0]
predicted_label = 1 if prediction >= 0.5 else 0

print("Predicted probability of positive review:", round(float(prediction), 4))
print("Predicted label:", "Positive" if predicted_label == 1 else "Negative")
print("Actual label:", "Positive" if actual_label == 1 else "Negative")

## 16. Save the LSTM Model

Saving the model lets us reuse it later without training again.

In [ ]:
lstm_model.save("day_10_beginner_lstm_imdb_model.keras")
print("Model saved successfully.")

## Beginner Revision Questions

1. What is sentiment classification?
2. Why do we convert words into numbers?
3. Why do we use padding?
4. What does the embedding layer do?
5. What is the main difference between RNN and LSTM?
6. Which model performed better in your run: RNN or LSTM?